# Support Ticket Classification & Prioritization Using ML & NLP
## Phase 1: Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('../dataset/sample_tickets.csv')
display(df.head())

# Show dataset info and missing values
print(df.info())
print("\nMissing amounts:\n", df.isnull().sum())

# Drop missing ranges
df.dropna(inplace=True)

# Visualize class distributions
plt.figure(figsize=(10,4))
sns.countplot(data=df, x='Category')
plt.title('Distribution of Categories')
plt.show()

plt.figure(figsize=(10,4))
sns.countplot(data=df, x='Priority')
plt.title('Distribution of Priorities')
plt.show()

## Phase 2: NLP Preprocessing

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['Clean_Text'] = df['Ticket Text'].apply(preprocess_text)
display(df[['Ticket Text', 'Clean_Text']].head())

## Phase 3: Feature Engineering
We will use TF-IDF Vectorization.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['Clean_Text'])
print("TF-IDF Matrix Shape:", X_tfidf.shape)

# Bag of Words (for comparison)
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['Clean_Text'])
print("Bag of Words Matrix Shape:", X_bow.shape)

## Phase 4 & 5: Category & Priority Classification (Model Training & Evaluation)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Split data
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_tfidf, df['Category'], test_size=0.2, random_state=42)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_tfidf, df['Priority'], test_size=0.2, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(),
    'Multinomial NB': MultinomialNB(),
    'Random Forest': RandomForestClassifier(),
    'SVM': SVC()
}

print("\n--- Category Classification Models ---")
best_cat_model = None
best_acc = 0
for name, model in models.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    acc = accuracy_score(y_test_c, preds)
    print(f"{name} Accuracy: {acc:.4f}")
    if acc >= best_acc:
        best_acc = acc
        best_cat_model = model

print("\n--- Priority Classification Models ---")
best_pri_model = None
best_p_acc = 0
for name, model in models.items():
    model.fit(X_train_p, y_train_p)
    preds = model.predict(X_test_p)
    acc = accuracy_score(y_test_p, preds)
    print(f"{name} Accuracy: {acc:.4f}")
    if acc >= best_p_acc:
        best_p_acc = acc
        best_pri_model = model

## Phase 6: Evaluation

In [ ]:
print("Best Category Model Evaluation:\n")
preds = best_cat_model.predict(X_test_c)
print(classification_report(y_test_c, preds))

cm = confusion_matrix(y_test_c, preds)
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix - Category')
plt.show()

## Phase 7 & 8: Prediction System & Model Saving

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
# Save Models
joblib.dump(best_cat_model, '../models/category_model.pkl')
joblib.dump(best_pri_model, '../models/priority_model.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

def predict_ticket(text):
    cleaned = preprocess_text(text)
    vec = tfidf.transform([cleaned])
    cat = best_cat_model.predict(vec)[0]
    pri = best_pri_model.predict(vec)[0]
    return {'Category': cat, 'Priority': pri}

sample = "My payment was deducted twice."
print("Sample Output:", predict_ticket(sample))